# Phoneme-Space Dependency Distance Analysis Across 37 Languages

This notebook demonstrates the **Phoneme-Interval Distance (PID)** metric, which recomputes
dependency distances in phoneme-space across Universal Dependencies treebanks.

**Research question:** Does phonological density predict Dependency Length Minimization (DLM)
strength when distance is measured in phonemes rather than words?

**Key finding:** A weak negative correlation (r=-0.324, p=0.051) between phonological density
and mean PID suggests languages with higher phonological density may exhibit marginally stronger
dependency length minimization in phoneme-space. The rank correlation between word-based and
phoneme-based distances is moderate (r=0.523, p=0.001), indicating phoneme-space provides a
meaningfully different but correlated view of syntactic distance.

The notebook loads pre-computed results from a demo dataset covering 14 languages from
9 language families, and reproduces the statistical analysis and visualizations.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3',
         'matplotlib==3.10.0', 'seaborn==0.13.2')

# No additional non-Colab packages needed for this notebook

In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import spearmanr

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# NumPy 2.0 compatibility shims (for older non-Colab packages)
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

print("All imports successful.")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-45370e-phonotactic-constraint-on-dependency/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL, falling back to local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    import os
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError(
        "Could not load mini_demo_data.json from GitHub or local filesystem. "
        "Make sure the file is in the working directory."
    )

In [ ]:
data = load_data()
print(f"Loaded data: {data['metadata']['n_languages']} languages, "
      f"{data['metadata']['n_families']} families")
print(f"Method: {data['metadata']['method_name']}")
print(f"Max sentences per language: {data['metadata']['max_sentences_per_language']}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
# All tunable parameters for the demo. Start with minimum values that produce output.

# Number of languages to include in analysis (from the loaded dataset)
# Original: 37 languages. Demo uses all 14 available.
N_LANGUAGES = 14  # Set to None to use all available

# Minimum languages needed for statistical analysis
MIN_LANGUAGES_FOR_STATS = 10

# Minimum languages needed for figures
MIN_LANGUAGES_FOR_FIGURES = 5

# Figure output settings
FIGURE_DPI = 150
SAVE_FIGURES = True  # Save figures to disk

# Original experiment parameters (for reference)
# MAX_SENTENCES_PER_LANGUAGE = 200  # Used in the full experiment
# ALL_CONFIG_IDS = 37 languages from UD treebanks

print(f"Config: N_LANGUAGES={N_LANGUAGES}, MIN_STATS={MIN_LANGUAGES_FOR_STATS}")

## Phoneme Estimation

The core innovation of this method is estimating phoneme counts from written text using
language-family-specific ratios. Each language family has a characteristic ratio of phonemes
per character, reflecting the writing system's efficiency.

In [ ]:
# ── Phoneme estimation (language-specific ratios) ─────────────────────────
# Average phonemes per character for different language families
PHONEME_RATIO = {
    # Indo-European (Latin script)
    "eng": 1.2, "deu": 1.1, "fra": 1.3, "spa": 1.1, "ita": 1.1,
    "nld": 1.1, "pol": 1.2, "ron": 1.1, "swe": 1.1, "dan": 1.1,
    "nor": 1.1, "ces": 1.1, "slk": 1.1, "slv": 1.1, "hrv": 1.1,
    # Indo-European (Cyrillic)
    "rus": 0.8, "ukr": 0.8, "bul": 0.8, "srp": 0.8,
    # Uralic
    "fin": 1.0, "hun": 1.0, "est": 1.0,
    # Turkic
    "tur": 1.0, "kaz": 0.9, "uzb": 0.9,
    # Afroasiatic
    "ara": 0.9, "heb": 0.8, "amh": 0.7, "hau": 1.0,
    # Indo-Aryan
    "hin": 0.6, "mar": 0.6,
    # Sino-Tibetan
    "zho": 1.0, "mya": 0.9, "tam": 0.8,
    # Austronesian
    "tgl": 1.1, "jav": 1.0, "ind": 1.0, "msa": 1.0,
    # Japonic/Koreanic
    "jpn": 0.7, "kor": 0.6,
    # Austroasiatic
    "vie": 0.9,
    # Kartvelian
    "kat": 1.0,
}

# Characters that don't count as phonemes (punctuation, whitespace)
NON_PHONEME_CHARS = set(' .,;:!?()[]{}"\'"\'-/\\|@#$%^&*+=<>~`')


def estimate_phoneme_count(word: str, lang_code: str) -> int:
    """Estimate phoneme count for a word using language-specific ratio."""
    # Clean the word
    clean = ''.join(c for c in word if c not in NON_PHONEME_CHARS)
    if not clean:
        return 0
    
    ratio = PHONEME_RATIO.get(lang_code, 1.0)
    # For CJK, use character count directly (each character ≈ 1 syllable)
    if any('\u4e00' <= c <= '\u9fff' for c in clean):
        return len(clean)
    if any('\u3040' <= c <= '\u309f' or '\u30a0' <= c <= '\u30ff' for c in clean):
        return len(clean)
    if any('\uac00' <= c <= '\ud7a3' for c in clean):
        return len(clean) // 3  # Korean syllables
    
    return max(1, int(len(clean) * ratio))


def get_language_code(config_id: str) -> str:
    """Extract language code from config_id."""
    return config_id.split('_')[0]


# Demonstrate on sample words
demo_words = {
    "eng": ["the", "dependency", "phoneme", "language"],
    "deu": ["der", "Abhaengigkeit", "Phonem", "Sprache"],
    "rus": ["\u0438", "\u0437\u0430\u0432\u0438\u0441\u0438\u043c\u043e\u0441\u0442\u044c", "\u0444\u043e\u043d\u0435\u043c\u0430", "\u044f\u0437\u044b\u043a"],
    "zho": ["\u4f8b", "\u4f9d\u8d56", "\u97f3\u7d20", "\u8bed\u8a00"],
    "jpn": ["\u306b", "\u4f9d\u5b58", "\u97f3\u7d20", "\u8a9e"],
}

print("Phoneme estimation examples:")
print(f"{'Language':<12} {'Word':<25} {'Est. Phonemes':<15}")
print("-" * 55)
for lang, words in demo_words.items():
    for word in words:
        count = estimate_phoneme_count(word, lang)
        print(f"{lang:<12} {word:<25} {count:<15}")

## Phoneme-Interval Distance (PID) Computation

The PID metric sums the phoneme counts of all tokens **strictly between** the head and
dependent in each dependency arc. This contrasts with the standard word-based dependency
distance, which simply counts the number of words between them.

Formula: `PID = sum(phoneme_counts[k])` for all `k` strictly between head and dependent positions.

In [ ]:
def compute_pid_for_sentence(
    tokens: list[str],
    heads: list[int],
    lang_code: str
) -> tuple[list[int], list[int]]:
    """Compute Phoneme-Interval Distance for each dependency arc.
    
    PID = total number of phonemes strictly between head and dependent.
    
    Returns:
        (pid_values, word_distances)
    """
    pid_values = []
    word_distances = []
    
    # Estimate phoneme counts for all tokens
    phoneme_counts = [estimate_phoneme_count(t, lang_code) for t in tokens]
    
    for j, h in enumerate(heads):
        if h == 0:  # Skip root
            continue
        
        head_pos = h - 1  # Convert to 0-indexed
        dep_pos = j
        
        # Word distance
        word_dist = abs(head_pos - dep_pos)
        word_distances.append(word_dist)
        
        # PID: sum of phoneme counts strictly between head and dependent
        min_pos = min(head_pos, dep_pos)
        max_pos = max(head_pos, dep_pos)
        
        pid = sum(phoneme_counts[k] for k in range(min_pos + 1, max_pos))
        pid_values.append(pid)
    
    return pid_values, word_distances


# Demo: compute PID on a sample English sentence
# "The quick brown fox jumps over the lazy dog"
# Dependencies: fox(4) -> the(1), quick(2), brown(3), jumps(5), over(6), lazy(8), dog(9)
# Simplified: head of each word is the previous word (for demonstration)
demo_tokens = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
demo_heads = [0, 4, 4, 5, 0, 5, 8, 8, 5]  # fox->the,quick,brown; jumps->fox,over; dog->the,lazy

pids, wds = compute_pid_for_sentence(demo_tokens, demo_heads, "eng")

print("Demo sentence: 'The quick brown fox jumps over the lazy dog'")
print(f"\n{'Word':<10} {'Head':<6} {'Word Dist':<12} {'PID':<6}")
print("-" * 40)
for i, (token, pid, wd) in enumerate(zip(demo_tokens, pids, wds)):
    head_idx = demo_heads[i]
    head_word = demo_tokens[head_idx - 1] if head_idx > 0 else "ROOT"
    print(f"{token:<10} {head_word:<6} {wd:<12} {pid:<6}")

print(f"\nMean word distance: {np.mean(wds):.2f}")
print(f"Mean PID: {np.mean(pids):.2f}")
print(f"PID/Word ratio: {np.mean(pids)/np.mean(wds):.2f}")

## Loading Pre-computed Results

The full experiment processes 37 UD treebanks, loading data from HuggingFace and computing
PIDs for up to 200 sentences per language. In this demo, we load the pre-computed results
and reproduce the statistical analysis and visualizations.

In [ ]:
# Extract language results from the loaded data
language_results = []
for ds in data['datasets']:
    ex = ds['examples'][0]  # First example has full metrics
    inp = json.loads(ex['input'])
    out = json.loads(ex['output'])
    
    language_results.append({
        'config_id': inp['config_id'],
        'language_name': inp['language'],
        'family': inp['family'],
        'word_order': inp['word_order'],
        'phonological_density': inp['phonological_density'],
        'phoneme_inventory_size': inp['phoneme_inventory_size'],
        'n_sentences': inp['n_sentences'],
        'n_dependencies': inp['n_dependencies'],
        'mean_word_distance': out['mean_word_distance'],
        'mean_pid': out['mean_pid'],
        'median_pid': out['median_pid'],
        'std_pid': out['std_pid'],
        'max_pid': out['max_pid'],
    })

# Apply N_LANGUAGES limit if set
if N_LANGUAGES is not None and len(language_results) > N_LANGUAGES:
    language_results = language_results[:N_LANGUAGES]

print(f"Loaded {len(language_results)} languages for analysis")
print(f"Families: {sorted(set(r['family'] for r in language_results))}")
print(f"Word orders: {sorted(set(r['word_order'] for r in language_results))}")

## Cross-Linguistic Statistical Analysis

We compute Spearman rank correlations between phonological density and both
phoneme-space and word-space dependency distances, plus a linear regression and
rank-rank correlation between the two distance measures.

In [ ]:
def run_analysis(language_results: list[dict]) -> dict:
    """Run statistical analysis on language results."""
    if len(language_results) < MIN_LANGUAGES_FOR_STATS:
        return {"error": f"Need at least {MIN_LANGUAGES_FOR_STATS} languages for analysis"}
    
    phon_density = [r['phonological_density'] for r in language_results]
    mean_pids = [r['mean_pid'] for r in language_results]
    mean_word_dists = [r['mean_word_distance'] for r in language_results]
    
    # Spearman correlations
    corr_pid, p_pid = spearmanr(phon_density, mean_pids)
    corr_wd, p_wd = spearmanr(phon_density, mean_word_dists)
    
    # Linear regression
    phon_density_arr = np.array(phon_density)
    mean_pids_arr = np.array(mean_pids)
    
    z = np.polyfit(phon_density, mean_pids, 1)
    predictions = np.poly1d(z)(phon_density_arr)
    ss_res = np.sum((mean_pids_arr - predictions) ** 2)
    ss_tot = np.sum((mean_pids_arr - np.mean(mean_pids_arr)) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    # Rank correlation between word distance and PID
    word_ranks = np.argsort(np.argsort(mean_word_dists)) + 1
    pid_ranks = np.argsort(np.argsort(mean_pids)) + 1
    rank_corr, rank_p = spearmanr(word_ranks, pid_ranks)
    
    return {
        "spearman_correlation_phoneme_space": {
            "r": float(corr_pid),
            "p_value": float(p_pid),
            "n": len(language_results)
        },
        "spearman_correlation_baseline": {
            "r": float(corr_wd),
            "p_value": float(p_wd),
            "n": len(language_results)
        },
        "regression": {
            "r_squared": float(r_squared),
            "slope": float(z[0]),
            "intercept": float(z[1])
        },
        "rank_correlation": {
            "r": float(rank_corr),
            "p_value": float(rank_p)
        }
    }


# Run analysis
correlations = run_analysis(language_results)

if "error" in correlations:
    print(f"Analysis skipped: {correlations['error']}")
else:
    print("=" * 60)
    print("STATISTICAL ANALYSIS RESULTS")
    print("=" * 60)
    
    pid_corr = correlations['spearman_correlation_phoneme_space']
    print(f"\nPhoneme-space correlation (density vs mean PID):")
    print(f"  Spearman r = {pid_corr['r']:.4f}, p = {pid_corr['p_value']:.4f}, n = {pid_corr['n']}")
    
    wd_corr = correlations['spearman_correlation_baseline']
    print(f"\nBaseline correlation (density vs mean word distance):")
    print(f"  Spearman r = {wd_corr['r']:.4f}, p = {wd_corr['p_value']:.4f}, n = {wd_corr['n']}")
    
    reg = correlations['regression']
    print(f"\nLinear regression (density -> mean PID):")
    print(f"  R² = {reg['r_squared']:.4f}")
    print(f"  Slope = {reg['slope']:.4f}")
    print(f"  Intercept = {reg['intercept']:.4f}")
    
    rank = correlations['rank_correlation']
    print(f"\nRank correlation (word distance vs PID):")
    print(f"  Spearman r = {rank['r']:.4f}, p = {rank['p_value']:.4f}")

## Visualization

Generate the five publication-quality figures from the original experiment:
1. Phonological density vs mean PID (scatter with trend line)
2. Phonological density vs word distance (baseline)
3. PID distribution by word order (boxplots)
4. Mean PID by language family (bar chart)
5. Rank-rank correlation plot

In [ ]:
def create_figures(language_results: list[dict]) -> list[str]:
    """Generate visualization figures from language results."""
    figures = []
    output_dir = Path("figures")
    output_dir.mkdir(exist_ok=True)
    
    if len(language_results) < MIN_LANGUAGES_FOR_FIGURES:
        print(f"Insufficient data for figures (need {MIN_LANGUAGES_FOR_FIGURES}, have {len(language_results)})")
        return figures
    
    lang_names = [r['config_id'] for r in language_results]
    mean_pids = [r['mean_pid'] for r in language_results]
    mean_word_dists = [r['mean_word_distance'] for r in language_results]
    families = [r['family'] for r in language_results]
    word_orders = [r['word_order'] for r in language_results]
    phon_density = [r['phonological_density'] for r in language_results]
    n_sents = [r['n_sentences'] for r in language_results]
    
    # Color map for families
    family_colors = sns.color_palette("husl", max(1, len(set(families))))
    family_to_color = dict(zip(sorted(set(families)), family_colors))
    
    # Figure 1: Phonological density vs mean PID
    fig, ax = plt.subplots(figsize=(10, 8))
    for i, (pd_val, pid_val, family, size) in enumerate(zip(phon_density, mean_pids, families, n_sents)):
        color = family_to_color.get(family, "gray")
        ax.scatter(pd_val, pid_val, c=color, s=max(20, size/3), alpha=0.7,
                   edgecolors='black', linewidth=0.5)
    
    if len(phon_density) > 5:
        z = np.polyfit(phon_density, mean_pids, 1)
        p = np.poly1d(z)
        x_range = np.linspace(min(phon_density), max(phon_density), 100)
        ax.plot(x_range, p(x_range), "r--", alpha=0.5, 
                label=f"r = {np.corrcoef(phon_density, mean_pids)[0,1]:.3f}")
    
    ax.set_xlabel("Phonological Density (phonemes/syllable)", fontsize=12)
    ax.set_ylabel("Mean Phoneme-Interval Distance", fontsize=12)
    ax.set_title("Phonological Density vs Dependency Distance in Phoneme-Space", fontsize=14)
    ax.legend()
    plt.tight_layout()
    fig_path = output_dir / "fig1_density_vs_pid.pdf"
    fig.savefig(fig_path, dpi=FIGURE_DPI, bbox_inches='tight')
    fig.savefig(output_dir / "fig1_density_vs_pid.png", dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)
    figures.append(str(fig_path))
    print(f"Saved {fig_path}")
    
    # Figure 2: Phonological density vs mean word distance (baseline)
    fig, ax = plt.subplots(figsize=(10, 8))
    for i, (pd_val, wd_val, family) in enumerate(zip(phon_density, mean_word_dists, families)):
        color = family_to_color.get(family, "gray")
        ax.scatter(pd_val, wd_val, c=color, s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    if len(phon_density) > 5:
        z = np.polyfit(phon_density, mean_word_dists, 1)
        p = np.poly1d(z)
        x_range = np.linspace(min(phon_density), max(phon_density), 100)
        ax.plot(x_range, p(x_range), "r--", alpha=0.5,
                label=f"r = {np.corrcoef(phon_density, mean_word_dists)[0,1]:.3f}")
    
    ax.set_xlabel("Phonological Density", fontsize=12)
    ax.set_ylabel("Mean Word Distance (baseline)", fontsize=12)
    ax.set_title("Phonological Density vs Word-Space Dependency Distance", fontsize=14)
    ax.legend()
    plt.tight_layout()
    fig_path = output_dir / "fig2_density_vs_word_distance.pdf"
    fig.savefig(fig_path, dpi=FIGURE_DPI, bbox_inches='tight')
    fig.savefig(output_dir / "fig2_density_vs_word_distance.png", dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)
    figures.append(str(fig_path))
    print(f"Saved {fig_path}")
    
    # Figure 3: PID distribution by word order
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for idx, wo in enumerate(["SVO", "SOV"]):
        mask = [w == wo for w in word_orders]
        if sum(mask) > 0:
            data = [mean_pids[i] for i, m in enumerate(mask) if m]
            bp = axes[idx].boxplot(data)
            axes[idx].set_xticklabels([wo])
            axes[idx].set_title(f"PID Distribution ({wo})", fontsize=12)
            axes[idx].set_ylabel("Mean PID")
    plt.tight_layout()
    fig_path = output_dir / "fig3_pid_by_word_order.pdf"
    fig.savefig(fig_path, dpi=FIGURE_DPI, bbox_inches='tight')
    fig.savefig(output_dir / "fig3_pid_by_word_order.png", dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)
    figures.append(str(fig_path))
    print(f"Saved {fig_path}")
    
    # Figure 4: Mean PID by family
    fig, ax = plt.subplots(figsize=(12, 6))
    family_means = {}
    for f, pid in zip(families, mean_pids):
        family_means.setdefault(f, []).append(pid)
    family_avg = {f: np.mean(pids) for f, pids in family_means.items()}
    bars = ax.bar(range(len(family_avg)), list(family_avg.values()),
                  color=[family_to_color.get(f, "gray") for f in family_avg.keys()])
    ax.set_xticks(range(len(family_avg)))
    ax.set_xticklabels(family_avg.keys(), rotation=45, ha='right', fontsize=9)
    ax.set_ylabel("Mean PID")
    ax.set_title("Mean Phoneme-Interval Distance by Language Family", fontsize=14)
    plt.tight_layout()
    fig_path = output_dir / "fig4_pid_by_family.pdf"
    fig.savefig(fig_path, dpi=FIGURE_DPI, bbox_inches='tight')
    fig.savefig(output_dir / "fig4_pid_by_family.png", dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)
    figures.append(str(fig_path))
    print(f"Saved {fig_path}")
    
    # Figure 5: Rank-rank plot
    fig, ax = plt.subplots(figsize=(10, 8))
    word_ranks = np.argsort(np.argsort(mean_word_dists)) + 1
    pid_ranks = np.argsort(np.argsort(mean_pids)) + 1
    ax.scatter(word_ranks, pid_ranks, s=50, alpha=0.6, edgecolors='black')
    max_rank = max(max(word_ranks), max(pid_ranks))
    ax.plot([1, max_rank], [1, max_rank], "r--", alpha=0.5, label="Identity")
    corr_r, _ = spearmanr(word_ranks, pid_ranks)
    ax.set_xlabel("Rank by Word Distance", fontsize=12)
    ax.set_ylabel("Rank by Phoneme-Interval Distance", fontsize=12)
    ax.set_title(f"Rank Correlation: Word Distance vs PID\n(Spearman r = {corr_r:.3f})", fontsize=14)
    ax.legend()
    plt.tight_layout()
    fig_path = output_dir / "fig5_rank_correlation.pdf"
    fig.savefig(fig_path, dpi=FIGURE_DPI, bbox_inches='tight')
    fig.savefig(output_dir / "fig5_rank_correlation.png", dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)
    figures.append(str(fig_path))
    print(f"Saved {fig_path}")
    
    return figures


# Generate figures
figures = create_figures(language_results)
print(f"\nGenerated {len(figures)} figures")

## Key Findings and Hypothesis Test

Summary of the statistical results and the hypothesis test outcome.

In [ ]:
# ── Results Summary ────────────────────────────────────────────────────────

# Build results table
df = pd.DataFrame(language_results)
df = df[['language_name', 'family', 'word_order', 'phonological_density', 
         'mean_word_distance', 'mean_pid', 'median_pid', 'std_pid', 'max_pid',
         'n_sentences', 'n_dependencies']]
df['pid_to_word_ratio'] = df['mean_pid'] / df['mean_word_distance']
df = df.sort_values('mean_pid')

print("=" * 80)
print("LANGUAGE RESULTS TABLE")
print("=" * 80)
print(df.to_string(index=False))

# Hypothesis test
print("\n" + "=" * 80)
print("HYPOTHESIS TEST")
print("=" * 80)

corr_info = correlations.get('spearman_correlation_phoneme_space', {})
r_value = corr_info.get('r', 0)
p_value = corr_info.get('p_value', 1.0)

if abs(r_value) < 0.3 and p_value > 0.05:
    conclusion = "null_result"
    hypothesis_support = "Phonological density does NOT significantly predict DLM strength in phoneme-space"
else:
    conclusion = "significant"
    hypothesis_support = f"Phonological density shows {'negative' if r_value < 0 else 'positive'} correlation with DLM strength (r={r_value:.3f}, p={p_value:.4f})"

print(f"Prediction: Phonological density is independent of DLM strength")
print(f"Result: {conclusion}")
print(f"Correlation (r): {r_value:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Conclusion: {hypothesis_support}")

# Compare with full experiment results
print("\n" + "=" * 80)
print("COMPARISON: Demo (14 langs) vs Full Experiment (37 langs)")
print("=" * 80)
full_corr = data['correlations']['spearman_correlation_phoneme_space']
print(f"{'Metric':<40} {'Demo (14 langs)':<20} {'Full (37 langs)':<20}")
print("-" * 80)
print(f"{'Spearman r (phoneme-space)':<40} {r_value:<20.4f} {full_corr['r']:<20.4f}")
print(f"{'p-value':<40} {p_value:<20.6f} {full_corr['p_value']:<20.6f}")

full_rank = data['correlations']['rank_correlation']
demo_rank = correlations.get('rank_correlation', {})
print(f"{'Rank correlation (word vs PID)':<40} {demo_rank.get('r', 0):<20.4f} {full_rank['r']:<20.4f}")

full_reg = data['correlations']['regression']
demo_reg = correlations.get('regression', {})
print(f"{'R-squared':<40} {demo_reg.get('r_squared', 0):<20.4f} {full_reg['r_squared']:<20.4f}")

print("\n" + "=" * 80)
print("NOTE: The demo uses 14 curated languages. The full experiment uses 37 languages.")
print("Correlation values may differ slightly due to the smaller sample.")
print("=" * 80)

In [ ]:
# Display generated figures inline
from IPython.display import display, Image, Markdown

figure_titles = [
    "Fig 1: Phonological Density vs Mean PID",
    "Fig 2: Phonological Density vs Word Distance (Baseline)",
    "Fig 3: PID Distribution by Word Order",
    "Fig 4: Mean PID by Language Family",
    "Fig 5: Rank Correlation (Word Distance vs PID)",
]

for i, fig_path in enumerate(figures):
    png_path = fig_path.replace('.pdf', '.png')
    if Path(png_path).exists():
        Markdown(f"### {figure_titles[i]}")
        display(Image(filename=png_path, width=700))
    else:
        print(f"Figure {i+1} not found: {png_path}")